# 05 — Analyze & Query

Sample analytical queries against the Gold star schema. Demonstrates:
- Star schema joins (fact → dimensions)
- Materialized view queries
- Full lineage row counts (Bronze → Silver → Gold)
- Data quality checks
- Optional cleanup

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "ticketmaster_demo")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"

G = f"{UC_CATALOG}.{GOLD_SCHEMA}"
print(f"Querying: {G}")

Querying: alexander_booth.ticketmaster_demo_gold


## Top 10 Venues by Event Count

In [2]:
spark.sql(f"""
    SELECT v.venue_name, v.city, v.state,
           COUNT(DISTINCT f.event_sk) AS event_count,
           ROUND(AVG(f.price_max), 2) AS avg_max_price
    FROM {G}.fact_events f
    INNER JOIN {G}.dim_venue v ON f.venue_sk_fk = v.venue_sk
    WHERE f.is_test = FALSE
    GROUP BY v.venue_name, v.city, v.state
    ORDER BY event_count DESC
    LIMIT 10
""").show(truncate=False)

+--------------------------------------+---------+---------+-----------+-------------+
|venue_name                            |city     |state    |event_count|avg_max_price|
+--------------------------------------+---------+---------+-----------+-------------+
|Treasure Island Resort & Casino       |Welch    |Minnesota|32         |NULL         |
|Yankee Stadium                        |Bronx    |New York |28         |NULL         |
|The Theater Center                    |New York |New York |28         |NULL         |
|The Loft                              |Atlanta  |Georgia  |24         |NULL         |
|Sphere                                |Las Vegas|Nevada   |20         |NULL         |
|Laugh Factory at Horseshoe Las Vegas  |Las Vegas|Nevada   |20         |NULL         |
|Mercury Lounge                        |New York |New York |16         |NULL         |
|Lexington Opera House                 |Lexington|Kentucky |16         |NULL         |
|Harrah's Cabaret at Harrah's Las Vegas|Las

## Events by Genre Breakdown

In [3]:
spark.sql(f"""
    SELECT c.segment_name, c.genre_name,
           COUNT(DISTINCT f.event_sk) AS event_count,
           ROUND(AVG(f.price_min), 2) AS avg_min_price,
           ROUND(AVG(f.price_max), 2) AS avg_max_price
    FROM {G}.fact_events f
    INNER JOIN {G}.dim_classification c ON f.classification_sk_fk = c.classification_sk
    WHERE f.is_test = FALSE
    GROUP BY c.segment_name, c.genre_name
    ORDER BY event_count DESC
    LIMIT 15
""").show(truncate=False)

+--------------+----------+-----------+-------------+-------------+
|segment_name  |genre_name|event_count|avg_min_price|avg_max_price|
+--------------+----------+-----------+-------------+-------------+
|Undefined     |NULL      |1252       |NULL         |NULL         |
|Arts & Theatre|NULL      |746        |NULL         |NULL         |
|Music         |NULL      |604        |NULL         |NULL         |
|Sports        |NULL      |448        |NULL         |NULL         |
|Miscellaneous |NULL      |290        |NULL         |NULL         |
+--------------+----------+-----------+-------------+-------------+



## Weekend vs Weekday Event Distribution

In [4]:
spark.sql(f"""
    SELECT
        CASE WHEN d.is_weekend THEN 'Weekend' ELSE 'Weekday' END AS day_type,
        COUNT(DISTINCT f.event_sk) AS event_count,
        ROUND(AVG(f.price_max), 2) AS avg_max_price,
        COUNT(DISTINCT f.venue_sk_fk) AS unique_venues
    FROM {G}.fact_events f
    INNER JOIN {G}.dim_date d ON f.date_sk_fk = d.date_sk
    WHERE f.is_test = FALSE
    GROUP BY CASE WHEN d.is_weekend THEN 'Weekend' ELSE 'Weekday' END
""").show(truncate=False)

+--------+-----------+-------------+-------------+
|day_type|event_count|avg_max_price|unique_venues|
+--------+-----------+-------------+-------------+
|Weekday |3226       |51.59        |927          |
|Weekend |1508       |53.47        |538          |
+--------+-----------+-------------+-------------+



## Price Range Analysis by Classification

In [5]:
spark.sql(f"""
    SELECT c.segment_name, c.type_name,
           COUNT(DISTINCT f.event_sk) AS events,
           ROUND(MIN(f.price_min), 2) AS min_price,
           ROUND(AVG((f.price_min + f.price_max) / 2), 2) AS avg_price,
           ROUND(MAX(f.price_max), 2) AS max_price
    FROM {G}.fact_events f
    INNER JOIN {G}.dim_classification c ON f.classification_sk_fk = c.classification_sk
    WHERE f.is_test = FALSE AND f.price_min IS NOT NULL
    GROUP BY c.segment_name, c.type_name
    ORDER BY avg_price DESC
    LIMIT 10
""").show(truncate=False)

+------------+---------+------+---------+---------+---------+
|segment_name|type_name|events|min_price|avg_price|max_price|
+------------+---------+------+---------+---------+---------+
+------------+---------+------+---------+---------+---------+



## Monthly Event Trends (Materialized View)

In [6]:
spark.sql(f"""
    SELECT year, month, month_name, total_events, unique_venues,
           unique_attractions, weekend_events, weekday_events,
           ROUND(avg_min_price, 2) AS avg_min_price,
           ROUND(avg_max_price, 2) AS avg_max_price
    FROM {G}.v_monthly_summary
    ORDER BY year, month
""").show(truncate=False)

+----+-----+----------+------------+-------------+------------------+--------------+--------------+-------------+-------------+
|year|month|month_name|total_events|unique_venues|unique_attractions|weekend_events|weekday_events|avg_min_price|avg_max_price|
+----+-----+----------+------------+-------------+------------------+--------------+--------------+-------------+-------------+
|2024|5    |May       |1           |1            |0                 |1             |0             |NULL         |NULL         |
|2024|9    |September |1           |0            |0                 |0             |1             |NULL         |NULL         |
|2024|10   |October   |6           |1            |1                 |0             |6             |NULL         |NULL         |
|2025|1    |January   |1           |0            |0                 |0             |1             |NULL         |NULL         |
|2025|3    |March     |3           |1            |1                 |3             |0             |NULL 

## Top Attractions by Geographic Spread (Materialized View)

In [7]:
spark.sql(f"""
    SELECT attraction_name, segment_name, genre_name,
           total_events, cities_count, states_count,
           first_event_date, last_event_date,
           ROUND(avg_max_price, 2) AS avg_max_price
    FROM {G}.v_events_by_attraction
    ORDER BY total_events DESC
    LIMIT 10
""").show(truncate=False)

+-------------------------------+--------------+----------+------------+------------+------------+----------------+---------------+-------------+
|attraction_name                |segment_name  |genre_name|total_events|cities_count|states_count|first_event_date|last_event_date|avg_max_price|
+-------------------------------+--------------+----------+------------+------------+------------+----------------+---------------+-------------+
|Brandon Lake                   |Music         |Rock      |4           |2           |2           |2026-03-19      |2026-04-19     |NULL         |
|Toronto Blue Jays              |Sports        |Baseball  |3           |1           |1           |2026-05-19      |2026-05-19     |NULL         |
|WWE                            |Sports        |Wrestling |3           |1           |1           |2026-04-19      |2026-04-19     |NULL         |
|New York Yankees               |Sports        |Baseball  |3           |1           |1           |2026-05-19      |2026-05-1

## Full Lineage: Row Counts Across Layers

In [8]:
print("=" * 60)
print("DATA LINEAGE — ROW COUNTS")
print("=" * 60)

layers = {
    "Bronze": [
        (f"{UC_CATALOG}.{BRONZE_SCHEMA}.events_raw", "events_raw"),
        (f"{UC_CATALOG}.{BRONZE_SCHEMA}.venues_raw", "venues_raw"),
        (f"{UC_CATALOG}.{BRONZE_SCHEMA}.attractions_raw", "attractions_raw"),
        (f"{UC_CATALOG}.{BRONZE_SCHEMA}.classifications_raw", "classifications_raw"),
    ],
    "Silver": [
        (f"{UC_CATALOG}.{SILVER_SCHEMA}.events", "events"),
        (f"{UC_CATALOG}.{SILVER_SCHEMA}.venues", "venues"),
        (f"{UC_CATALOG}.{SILVER_SCHEMA}.attractions", "attractions"),
        (f"{UC_CATALOG}.{SILVER_SCHEMA}.classifications", "classifications"),
        (f"{UC_CATALOG}.{SILVER_SCHEMA}.event_attractions", "event_attractions"),
    ],
    "Gold": [
        (f"{UC_CATALOG}.{GOLD_SCHEMA}.fact_events", "fact_events"),
        (f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_date", "dim_date"),
        (f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_venue", "dim_venue"),
        (f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_attraction", "dim_attraction"),
        (f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_classification", "dim_classification"),
        (f"{UC_CATALOG}.{GOLD_SCHEMA}.bridge_event_attractions", "bridge_event_attractions"),
    ],
}

for layer, tables in layers.items():
    print(f"\n{layer}:")
    for full_name, short_name in tables:
        count = spark.table(full_name).count()
        print(f"  {short_name:30s} {count:>8,}")

DATA LINEAGE — ROW COUNTS

Bronze:
  events_raw                        4,800
  venues_raw                        1,600
  attractions_raw                   1,600
  classifications_raw                  34

Silver:
  events                            4,800
  venues                            1,600
  attractions                       1,600
  classifications                      34
  event_attractions                 3,886

Gold:
  fact_events                       4,800
  dim_date                          1,461
  dim_venue                         1,600
  dim_attraction                    1,600
  dim_classification                   34
  bridge_event_attractions          3,886


## Data Quality Checks

In [9]:
print("=" * 60)
print("DATA QUALITY CHECKS")
print("=" * 60)

# 1. Null PK check
print("\n1. Null primary keys:")
pk_checks = [
    (f"{UC_CATALOG}.{GOLD_SCHEMA}.fact_events", "event_sk"),
    (f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_venue", "venue_sk"),
    (f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_attraction", "attraction_sk"),
    (f"{UC_CATALOG}.{GOLD_SCHEMA}.dim_classification", "classification_sk"),
]
for table, pk in pk_checks:
    nulls = spark.sql(f"SELECT COUNT(*) AS c FROM {table} WHERE {pk} IS NULL").collect()[0]["c"]
    print(f"  {table.split('.')[-1]}.{pk}: {'PASS' if nulls == 0 else f'FAIL ({nulls} nulls)'}")

# 2. Duplicate SK check
print("\n2. Duplicate surrogate keys:")
for table, pk in pk_checks:
    dupes = spark.sql(f"SELECT {pk}, COUNT(*) c FROM {table} GROUP BY {pk} HAVING c > 1").count()
    print(f"  {table.split('.')[-1]}.{pk}: {'PASS' if dupes == 0 else f'FAIL ({dupes} dupes)'}")

# 3. Orphaned FK check
print("\n3. Orphaned foreign keys:")
orphaned_venues = spark.sql(f"""
    SELECT COUNT(*) AS c FROM {UC_CATALOG}.{GOLD_SCHEMA}.fact_events f
    LEFT JOIN {UC_CATALOG}.{GOLD_SCHEMA}.dim_venue v ON f.venue_sk_fk = v.venue_sk
    WHERE f.venue_sk_fk IS NOT NULL AND v.venue_sk IS NULL
""").collect()[0]["c"]
print(f"  fact_events.venue_sk_fk: {'PASS' if orphaned_venues == 0 else f'FAIL ({orphaned_venues} orphans)'}")

orphaned_dates = spark.sql(f"""
    SELECT COUNT(*) AS c FROM {UC_CATALOG}.{GOLD_SCHEMA}.fact_events f
    LEFT JOIN {UC_CATALOG}.{GOLD_SCHEMA}.dim_date d ON f.date_sk_fk = d.date_sk
    WHERE f.date_sk_fk IS NOT NULL AND d.date_sk IS NULL
""").collect()[0]["c"]
print(f"  fact_events.date_sk_fk: {'PASS' if orphaned_dates == 0 else f'FAIL ({orphaned_dates} orphans)'}")

orphaned_class = spark.sql(f"""
    SELECT COUNT(*) AS c FROM {UC_CATALOG}.{GOLD_SCHEMA}.fact_events f
    LEFT JOIN {UC_CATALOG}.{GOLD_SCHEMA}.dim_classification c ON f.classification_sk_fk = c.classification_sk
    WHERE f.classification_sk_fk IS NOT NULL AND c.classification_sk IS NULL
""").collect()[0]["c"]
print(f"  fact_events.classification_sk_fk: {'PASS' if orphaned_class == 0 else f'FAIL ({orphaned_class} orphans)'}")

print("\nData quality checks complete.")

DATA QUALITY CHECKS

1. Null primary keys:
  fact_events.event_sk: PASS
  dim_venue.venue_sk: PASS
  dim_attraction.attraction_sk: PASS
  dim_classification.classification_sk: PASS

2. Duplicate surrogate keys:
  fact_events.event_sk: PASS
  dim_venue.venue_sk: PASS
  dim_attraction.attraction_sk: PASS
  dim_classification.classification_sk: PASS

3. Orphaned foreign keys:
  fact_events.venue_sk_fk: FAIL (3452 orphans)
  fact_events.date_sk_fk: PASS
  fact_events.classification_sk_fk: FAIL (3130 orphans)

Data quality checks complete.


## Cleanup (Optional)

Uncomment and run the cell below to drop all schemas and their tables.

In [10]:
# WARNING: This will delete ALL data created by this demo.
# Uncomment the lines below to run.

# for schema in [GOLD_SCHEMA, SILVER_SCHEMA, BRONZE_SCHEMA]:
#     spark.sql(f"DROP SCHEMA IF EXISTS {UC_CATALOG}.{schema} CASCADE")
#     print(f"Dropped: {UC_CATALOG}.{schema}")
# print("Cleanup complete.")